# TG-119: Code-Based Planning with Machine Learning Outcome Models

![gallery_thumbnail](_static/notebooks/code_tg119_ml/cover.png)

## Intro

Welcome to the <i>TG-119: Code-Based Planning with Machine Learning Outcome Models</i> notebook! <br><br> In this notebook, we will augment the <i>TG-119: Conventional Code-Based Planning</i> notebook with machine learning (ML) outcome models using data from the TG-119 standard case (available from our [Github repository's docs folder](https://github.com/pyanno4rt/pyanno4rt/tree/master/docs) as .mat- and .csv-files).

> Note: we will focus on the parts beyond conventional planning, and recommend checking out the other notebook if you haven't done yet!

## Import of the relevant classes

In [ ]:
from pyanno4rt.base import (
    Configuration, Evaluation, Optimization, TreatmentPlan)

## Treatment plan initialization

### Setting up the configuration object

In [ ]:
configuration = Configuration(
    label='TG-119-ml',  # Unique identifier for the treatment plan
    modality='photon',  # Treatment modality
    imaging_path='./TG_119_data.mat',  # Path to the CT and segmentation data
    dose_matrix_path='./TG_119_photonDij.mat',  # Path to the dose-influence matrix
    dose_resolution=[6, 6, 6],  # Size of the dose grid in [mm] per dimension
    min_log_level='info',  # Minimum logging level
    number_of_fractions=30  # Number of fractions
    )

Apart from the label, the configuration object must not be changed compared to the conventional TG-119 plan.

### Setting up the optimization object

#### Setting up a machine learning outcome model-based component

In our package, we have embedded the ML outcome models directly into the optimization components. By adding the respective component to the component list in the optimization object, these models become part of the decision process, making the solver balance outcome and conventional plan criteria. In the following, we will initialize an exemplary logistic regression outcome model-based component step by step.

> Note: the component initialization follows the same pattern for all implemented ML models, with the exception of the respective classes to be initialized for the hyperparameter search and the component itself. Please check out the <i>Optimization components</i> notebook in the [Notebooks](https://pyanno4rt.readthedocs.io/en/latest/notebooks.html) section for more information on the components.

##### Specifying the data path

In [ ]:
data_path = './TG_119_synthetic.csv'

To fit the ML model, we created a synthetic NTCP dataset from a series of conventional TG-119 plans evaluated with the Lyman-Kutcher-Burman NTCP model to estimate the label values. This dataset can be found under the path stated above.

##### Define the input features and the label

In [ ]:
from pyanno4rt.learning.features import DynamicFeature, Label

data_columns = [
    DynamicFeature(column='core_mean', segment='Core', function='Dose Mean'),  # Mean dose
    DynamicFeature(column='core_std', segment='Core', function='Dose Deviation'),  # Standard deviation
    DynamicFeature(column='core_min', segment='Core', function='Dose Minimum'),  # Minimum dose
    DynamicFeature(column='core_max', segment='Core', function='Dose Maximum'),  # Maximum dose
    Label(column='label')  # Binary label for classification
    ]

If you provide an outcome dataset, you must define the input features and the label to be taken into account as a list using the following classes:

> `pyanno4rt.learning._columns.DynamicFeature`<br>
> <i>represents a "dynamic" feature (e.g. mean dose), i.e., a feature which may be recalculated within each iteration using some function</i>

> `pyanno4rt.learning._columns.StaticFeature`<br>
> <i>represents a "static" feature (e.g. patient age), i.e., a feature which has a constant, unchanging value</i>

> `pyanno4rt.learning._columns.Label`<br>
> <i>represents a label (e.g. toxicity grade), which may be binarized and linked to a time after treatment variable</i>

Please check out the <i>Data columns</i> notebook in the [Notebooks](https://pyanno4rt.readthedocs.io/en/latest/notebooks.html) section for more information on the features and the label.

##### Definition of the hyperparameter search space

In [ ]:
from pyanno4rt.learning.tune_spaces import TuneSpaceLR

tune_space = TuneSpaceLR(
    C=[2**-5, 2**10],  # Inverse regularization strength
    penalty=['l1', 'l2', 'elasticnet'],  # Penalty function
    tol=[1e-4, 1e-5, 1e-6],  # Stopping criteria tolerance
    class_weight=[None, 'balanced']  # Weights associated with the classes
    )

Our package also supports automatic hyperparameter tuning via sequential model-based optimization with tree-structured Parzen estimators. For this purpose, each ML model is assigned a tune space object that determines the search spaces of the respective hyperparameters. In the case of the logistic regression model, we need to initialize an object of the `TuneSpaceLR` class.

##### Choice of the display options

In [ ]:
from pyanno4rt.learning.evaluation import DisplayOptions

display_options = DisplayOptions()

The quality of the fitted ML model is captured by a number of evaluation metrics, which can be displayed either in the form of graphs ('AUC-ROC', 'AUC-PR', 'F1') or tabulated values ('Logloss', 'Brier score', 'Subset accuracy', 'Cohen Kappa', 'Hamming loss', 'Jaccard score', 'Precision', 'Recall', 'F1 score', 'MCC', 'AUC'). You can choose which options should be displayed by initializing an object of the `DisplayOptions` class. Its default state covers all metrics, which is sufficient for this example notebook.

##### Compilation into the model parameter object

In [ ]:
from pyanno4rt.learning import ModelParameters

model_parameters = ModelParameters(
    model_label='lrNTCP',  # Label for the model
    model_type='logistic',  # Type of the model
    data_path=data_path,  # Path to the data set
    data_columns=data_columns,  # List of features and label
    preprocessing=['StandardScaler'],  # Preprocessing steps
    tune_space=tune_space,  # Hyperparameter search space
    tune_evaluations=50,  # Number of hyperparameter evaluations
    tune_score='AUC',  # Hyperparameter score function
    tune_splits=5,  # Number of k-fold CV splits in each evaluation
    tune_repeats=1,  # Number of k-fold CV repeats in each evaluation
    inspect=True,  # Indicator for model inspection
    evaluate=True,  # Indicator for model evaluation
    oof_splits=5,  # Number of k-fold CV splits for evaluation
    oof_repeats=1,  # Number of k-fold CV repeats for evaluation
    write_features=True,  # Indicator for feature tracking
    display_options=display_options  # Display options
    )

Finally, we can put everything together and initialize the model parameter object. The model label is only used as an identifier, while the model type should be 'logistic', referring to the logistic regression outcome model. Data path and data columns have already been specified, as have the tune space and the display options. There is also a preprocessing parameter with a list of preprocessing steps (in this case, only a standard scaling step). Other model parameters describe either the hyperparameter optimization process or the inspection/evaluation steps after model fitting.

#### Adding the machine learning outcome model-based optimization component

In [ ]:
from pyanno4rt.optimization.components import (
    LogisticRegressionNTCP, SquaredDeviation, SquaredOverdosing)

optimization = Optimization(
    components=[  # Optimization components for each segment of interest
        LogisticRegressionNTCP(segment='Core', model_parameters=model_parameters, weight=1),
        SquaredOverdosing(segment='Core', maximum_dose=25, weight=100),
        SquaredDeviation(segment='OuterTarget', target_dose=60, weight=1000),
        SquaredOverdosing(segment='BODY', maximum_dose=30, weight=800)],
    method='weighted-sum',  # Single- or multi-criteria optimization method
    solver='scipy',  # Python package to be used for solving the optimization problem
    algorithm='L-BFGS-B',  # Solution algorithm from the chosen solver
    initial_strategy='target-coverage',  # Initialization strategy for the fluence vector
    initial_fluence_vector=None,  # User-defined initial fluence vector (only for 'warm-start')
    lower_variable_bounds=0,  # Lower bounds on the decision variables
    upper_variable_bounds=None,  # Upper bounds on the decision variables
    maximum_iterations=500,  # Maximum number of iterations for the solvers to converge
    tolerance=0.001  # Precision goal for the objective function value
    )

Once we have defined the model parameters, we can add the ML model-based optimization component to the component list. This works like any conventional component, with the only difference that we insert model parameters instead of physical (dose) parameters. In this example notebook, the following class has been initialized:

> `pyanno4rt.optimization.components._logistic_regression_ntcp.LogisticRegressionNTCP`<br>
> <i>refers to a function that incorporates a logistic regression outcome model</i>

### Setting up the evaluation object

In [ ]:
evaluation = Evaluation(
    dvh_type='cumulative',  # Type of DVH to be calculated
    number_of_points=1000,  # Number of (evenly-spaced) points for which to evaluate the DVH
    reference_volume=[2, 5, 50, 95, 98],  # Reference volumes for which to calculate the inverse DVH values
    reference_dose=[],  # Reference dose values for which to calculate the DVH values
    display_segments=[],  # Names of the segmented structures to be displayed
    display_metrics=[]  # Names of the plan evaluation metrics to be displayed
    )

The evaluation object must not be changed compared to the conventional TG-119 plan.

### Initializing the base class

In [ ]:
tp = TreatmentPlan(configuration, optimization, evaluation)

## Treatment plan workflow

The workflow steps are also identical to the conventional TG-119 plan.

### Configuring the plan

In [ ]:
tp.configure()

### Modeling for the plan

In [ ]:
tp.model()

Unlike in the conventional case, the `model` method now triggers the outcome modeling pipeline, including dataset handling and preprocessing, hyperparameter optimization and model fitting, and (optionally) model inspection and evaluation. Afterwards, the ML model-based optimization component can access the model instance during the solver run.

### Optimizing the plan

In [ ]:
tp.optimize()

### Evaluating the plan

In [ ]:
tp.evaluate()

### Visualizing the plan

In [ ]:
tp.visualize()

The `visualize` method opens the visual analysis tool as before. This time, however, the outcome graph will become available, as well as the visualizations in the "Data-driven model review" tab (provided that the inspection and/or evaluation parameters have been set to True).

![pyanno4rt visualizer](_static/notebooks/code_tg119_ml/ntcp_graph.png)

## Outro

We hope that this little example illustrates the basic usage of the code-based *pyanno4rt* interface for machine learning outcome model-based treatment planning. If you have any remarks, please take a look at the [Help and Support](https://pyanno4rt.readthedocs.io/en/latest/help_support.html) section and drop us a line. We would also be happy if you leave a positive comment and recommend our work to others. <br><br> Thank you for using *pyanno4rt* 😊